In [9]:
# conda run -n henv python -m pip install "setuptools<81"
import xspec
import batanalysis as ba
from swifttools.swift_too import GUANO, ObsQuery, Clock
import swiftbat

import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
from pathlib import Path
from astropy.io import fits
from astropy.time import Time, TimeDelta
import astropy.units as u
from astropy.coordinates import SkyCoord
import datetime
import os

from matplotlib.lines import Line2D

/home/gevas/anaconda3/envs/henv/lib/python3.14/site-packages/batanalysis/mosaic.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


Initalizing the BatAnalysis package
Completed initalizing the package


In [10]:
tmp_download_dir="/tmp/batdata/download_examples"

ba.datadir(tmp_download_dir, mkdir=True)


PosixPath('/tmp/batdata/download_examples')

In [11]:
query = ObsQuery(targetid=1088940)
query

2021-12-11 13:10:37,2021-12-11 13:15:21,GRB 211211A,01088940000,215,69
2021-12-11 14:07:00,2021-12-11 14:33:53,GRB 211211A,01088940000,1520,93
2021-12-11 17:35:01,2021-12-11 18:00:57,GRB 211211A,01088940001,1515,41
2021-12-11 19:10:02,2021-12-11 19:19:36,GRB 211211A,01088940001,480,94
2021-12-12 04:47:02,2021-12-12 05:06:23,GRB 211211A,01088940002,1095,66
2021-12-12 06:18:01,2021-12-12 06:37:54,GRB 211211A,01088940002,1100,93
2021-12-12 07:50:02,2021-12-12 08:10:20,GRB 211211A,01088940002,1100,118
2021-12-12 09:28:02,2021-12-12 09:47:49,GRB 211211A,01088940002,1095,92
2021-12-12 11:02:02,2021-12-12 11:21:38,GRB 211211A,01088940002,1095,81
2021-12-12 14:17:02,2021-12-12 14:36:55,GRB 211211A,01088940002,1100,93
2021-12-12 16:05:02,2021-12-12 16:18:58,GRB 211211A,01088940002,760,76


In [12]:
query = ObsQuery(name="GRB211211A")
query

2021-12-11 13:10:37,2021-12-11 13:15:21,GRB 211211A,01088940000,215,69
2021-12-11 14:07:00,2021-12-11 14:33:53,GRB 211211A,01088940000,1520,93
2021-12-11 17:35:01,2021-12-11 18:00:57,GRB 211211A,01088940001,1515,41
2021-12-11 19:10:02,2021-12-11 19:19:36,GRB 211211A,01088940001,480,94
2021-12-12 04:47:02,2021-12-12 05:06:23,GRB 211211A,01088940002,1095,66
2021-12-12 06:18:01,2021-12-12 06:37:54,GRB 211211A,01088940002,1100,93
2021-12-12 07:50:02,2021-12-12 08:10:20,GRB 211211A,01088940002,1100,118
2021-12-12 09:28:02,2021-12-12 09:47:49,GRB 211211A,01088940002,1095,92
2021-12-12 11:02:02,2021-12-12 11:21:38,GRB 211211A,01088940002,1095,81
2021-12-12 14:17:02,2021-12-12 14:36:55,GRB 211211A,01088940002,1100,93
2021-12-12 16:05:02,2021-12-12 16:18:58,GRB 211211A,01088940002,760,76


In [ ]:
download = ba.download_swiftdata(observations=query[0].obsid)
print(download)

In [ ]:
event=ba.BatEvent(query[0].obsid)

In [ ]:
# dt = 64 * u.ms   # or np.timedelta64(64, "ms") if timedelta itself doesn't need units — check separately
dt = np.timedelta64(128, "ms")
lc_total = event.create_lightcurve(energybins=[15, 150,350] * u.keV, timebinalg="uniform", timedelta=np.timedelta64(64, "ms"),recalc=True)   # R1: 15-150
lc_soft  = event.create_lightcurve(energybins=[15, 50,350]  * u.keV, timebinalg="uniform",timedelta=dt,recalc=True)   # R2: 15-50
lc_hard  = event.create_lightcurve(energybins=[50, 150,350] * u.keV,timebinalg="uniform", timedelta=dt,recalc=True)   # R3: 50-150

In [ ]:
fig, ax =lc_total.plot(T0=event.trigtime.met, plot_relative=True)

In [ ]:
t0 = event.trigtime.met
time = lc_total.data["TIME"].value - t0
rate = lc_total.data["RATE"]      # check shape — may be 2D: (ntime, nchan)
err  = lc_total.data["ERROR"]

# if RATE is 2D, channel 0 = 15-150 keV (first bin edge pair)
if rate.ndim == 2:
    rate_15_150 = rate[:, 0]
    err_15_150  = err[:, 0]
else:
    rate_15_150 = rate
    err_15_150  = err

fig, ax = plt.subplots()
ax.errorbar(time, rate_15_150, yerr=err_15_150, fmt="-", ds="steps-mid",label='15-150 keV')


time = lc_soft.data["TIME"].value - t0
rate = lc_soft.data["RATE"]      # check shape — may be 2D: (ntime, nchan)
err  = lc_soft.data["ERROR"]

# if RATE is 2D, channel 0 = 15-150 keV (first bin edge pair)
if rate.ndim == 2:
    rate_15_50 = rate[:, 0]
    err_15_50  = err[:, 0]
else:
    rate_15_50 = rate
    err_15_50  = err


ax.errorbar(time, rate_15_50, yerr=err_15_50, fmt="-", ds="steps-mid",label='15-50 keV')

time = lc_hard.data["TIME"].value - t0
rate = lc_hard.data["RATE"]      # check shape — may be 2D: (ntime, nchan)
err  = lc_hard.data["ERROR"]

# if RATE is 2D, channel 0 = 15-150 keV (first bin edge pair)
if rate.ndim == 2:
    rate_50_150 = rate[:, 0]
    err_50_150  = err[:, 0]
else:
    rate_50_150 = rate
    err_50_150  = err


ax.errorbar(time, rate_50_150, yerr=err_50_150, fmt="-", ds="steps-mid",label='50-150 keV')


ax.set_xlim(-2, 65)
ax.set_xlabel("Time since T0 (s)")
ax.set_ylabel("Rate (cts/s/det)")

plt.legend()